# 03 — Multi-Scale Wind-Vector Feature Engineering

This notebook prepares the modeling dataset for the study:

**A Shamal-Transition-Aware Multi-Scale Residual Neural Network for Multi-Horizon Wind Vector Forecasting over the Persian Gulf and Gulf of Oman**

The input consists of the spatially preprocessed annual NetCDF files generated by `02_preprocessing.ipynb`. These files contain hourly 10-m zonal and meridional wind components (`u10` and `v10`) for the official 512-cell Persian Gulf–Gulf of Oman study grid.

This notebook changes the original scalar one-hour wind-speed forecasting formulation into a **causal multi-horizon wind-vector forecasting problem**.

For every forecast issue time `t`, the engineered dataset contains:

- historical `u10` and `v10` information available at or before `t`,
- temporal information represented with cyclic encodings,
- preserved spatial coordinates and study-grid identifiers,
- future vector targets `(u10, v10)` at 1, 3, 6, 12, and 24 hours ahead.

The historical wind information is organized into three candidate temporal scales:

- **short scale:** 0, 1, 2, 3, and 6 h before issue time,
- **diurnal scale:** 12, 24, and 48 h before issue time,
- **synoptic scale:** 72, 120, and 168 h before issue time.

The resulting lag dimension is designed so that the later neural network can route different temporal scales through separate residual branches.

No Shamal labels are created in this notebook. Shamal-event and transition identification should be implemented in the next dedicated notebook after the meteorological event definition has been fixed from the literature.

No feature scaling and no train/validation/test transformation is performed here. Scaling remains a later step and must be fitted only on the training period.


## 1. Imports and project paths

This section imports the libraries required for vector feature engineering and defines the project directories.

The notebook is designed to work when launched either from the project root or directly from the `notebooks/` directory.

Input annual datasets are read from:

`data/processed/yearly/`

The new vector feature datasets are written to:

`data/processed/vector_features/`

A separate output directory is used so the previous scalar wind-speed feature files are not silently overwritten.

Metadata tables are stored under:

`outputs/tables/`


In [9]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import xarray as xr


# ---------------------------------------------------------
# Locate the project root robustly
# ---------------------------------------------------------

CWD = Path.cwd()

if (CWD / "data").exists() and (CWD / "notebooks").exists():
    PROJECT_ROOT = CWD

elif CWD.name == "notebooks" and (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent

else:
    raise RuntimeError(
        "Could not locate the wind_paper project root. "
        "Run this notebook from the project root "
        "or from the notebooks directory."
    )


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

YEARLY_DIR = PROCESSED_DIR / "yearly"

VECTOR_FEATURE_DIR = PROCESSED_DIR / "vector_features"

STUDY_GRID_PATH = PROCESSED_DIR / "study_grid.csv"

SUMMARY_DIR = PROJECT_ROOT / "outputs" / "tables"


VECTOR_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project root:")
print(PROJECT_ROOT.resolve())

print("\nInput yearly datasets:")
print(YEARLY_DIR.resolve())

print("\nVector-feature output directory:")
print(VECTOR_FEATURE_DIR.resolve())

print("\nStudy grid:")
print(STUDY_GRID_PATH.resolve())

print("\nSummary directory:")
print(SUMMARY_DIR.resolve())


Project root:
/home/mike/Desktop/wind_paper

Input yearly datasets:
/home/mike/Desktop/wind_paper/data/processed/yearly

Vector-feature output directory:
/home/mike/Desktop/wind_paper/data/processed/vector_features

Study grid:
/home/mike/Desktop/wind_paper/data/processed/study_grid.csv

Summary directory:
/home/mike/Desktop/wind_paper/outputs/tables


## 2. Define the multi-scale, multi-horizon forecasting schema

The forecast issue time is denoted by `t`.

Only wind information available at or before `t` may be used as input. The model will predict the future 10-m wind vector:

\[
\hat{\mathbf{V}}_{t+h}
=
(\hat{u}_{10,t+h},\hat{v}_{10,t+h})
\]

for the forecast horizons:

\[
h \in \{1,3,6,12,24\}\ \mathrm{h}.
\]

The current wind components at issue time `t` are valid predictors because all targets occur strictly in the future.

Historical wind information is represented using three temporal scales. These are the initial candidate scales for the publication experiments and can later be assessed through validation and ablation analysis.

The maximum required historical context is 168 hours and the maximum future target horizon is 24 hours.


In [10]:
# ---------------------------------------------------------
# Multi-scale historical schema
# ---------------------------------------------------------

TEMPORAL_SCALES = {
    "short": [
        0,
        1,
        2,
        3,
        6,
    ],
    "diurnal": [
        12,
        24,
        48,
    ],
    "synoptic": [
        72,
        120,
        168,
    ],
}


# ---------------------------------------------------------
# Multi-horizon target schema
# ---------------------------------------------------------

FORECAST_HORIZONS = [
    1,
    3,
    6,
    12,
    24,
]


# ---------------------------------------------------------
# Derived configuration
# ---------------------------------------------------------

ALL_LAGS = sorted({
    lag
    for scale_lags in TEMPORAL_SCALES.values()
    for lag in scale_lags
})

LAG_TO_SCALE = {
    lag: scale_name
    for scale_name, scale_lags in TEMPORAL_SCALES.items()
    for lag in scale_lags
}

MAX_HISTORY_HOURS = max(ALL_LAGS)
MAX_FORECAST_HOURS = max(FORECAST_HORIZONS)


print("Historical temporal scales:")

for scale_name, lags in TEMPORAL_SCALES.items():
    print(
        f"- {scale_name:8s}: {lags}"
    )

print("\nAll historical lags:")
print(ALL_LAGS)

print("\nForecast horizons:")
print(FORECAST_HORIZONS)

print(
    "\nMaximum historical context:",
    f"{MAX_HISTORY_HOURS} hours"
)

print(
    "Maximum forecast horizon:",
    f"{MAX_FORECAST_HOURS} hours"
)

print(
    "\nHistorical vector values per grid-time sample:",
    len(ALL_LAGS) * 2
)

print(
    "Future vector targets per grid-time sample:",
    len(FORECAST_HORIZONS) * 2
)


Historical temporal scales:
- short   : [0, 1, 2, 3, 6]
- diurnal : [12, 24, 48]
- synoptic: [72, 120, 168]

All historical lags:
[0, 1, 2, 3, 6, 12, 24, 48, 72, 120, 168]

Forecast horizons:
[1, 3, 6, 12, 24]

Maximum historical context: 168 hours
Maximum forecast horizon: 24 hours

Historical vector values per grid-time sample: 22
Future vector targets per grid-time sample: 10


## 3. Validate the processed input datasets

Before engineering any new variables, the processed annual NetCDF files are checked.

This step verifies that:

- all 16 annual files from 2005 through 2020 are present,
- the official study grid contains 512 unique cells,
- each annual dataset contains `u10` and `v10`,
- the expected spatial coordinates are preserved,
- the temporal axis remains hourly.

This prevents feature engineering from continuing if the preprocessing stage is incomplete or inconsistent.


In [11]:
# ---------------------------------------------------------
# Validate study grid
# ---------------------------------------------------------

study_grid = pd.read_csv(
    STUDY_GRID_PATH
)

assert len(study_grid) == 512
assert study_grid["study_id"].nunique() == 512
assert study_grid["grid_id"].nunique() == 512

print("Study-grid validation: PASSED")

print(
    "Persian Gulf cells:",
    int((study_grid["basin"] == "Persian Gulf").sum())
)

print(
    "Gulf of Oman cells:",
    int((study_grid["basin"] == "Gulf of Oman").sum())
)


# ---------------------------------------------------------
# Validate annual processed files
# ---------------------------------------------------------

expected_years = list(
    range(2005, 2021)
)

yearly_files = sorted(
    YEARLY_DIR.glob("*.nc")
)

found_years = [
    int(file.stem)
    for file in yearly_files
]


assert found_years == expected_years, (
    "Processed annual files do not match the expected "
    "2005–2020 sequence."
)


for file in yearly_files:

    year = int(file.stem)

    with xr.open_dataset(file) as ds:

        assert "u10" in ds.data_vars
        assert "v10" in ds.data_vars

        assert "valid_time" in ds.dims
        assert "study_id" in ds.dims

        assert ds.sizes["study_id"] == 512

        times = pd.DatetimeIndex(
            pd.to_datetime(
                ds["valid_time"].values
            )
        )

        assert not times.has_duplicates

        time_differences = (
            pd.Series(times)
            .diff()
            .dropna()
        )

        assert (
            time_differences
            == pd.Timedelta(hours=1)
        ).all()


print(
    "\nProcessed annual input validation: PASSED"
)

print(
    "Annual files:",
    len(yearly_files)
)


Study-grid validation: PASSED
Persian Gulf cells: 357
Gulf of Oman cells: 155

Processed annual input validation: PASSED
Annual files: 16


## 4. Construct causal multi-scale wind-vector predictors and future targets

The feature-engineering function below creates two vector-history tensors:

- `u10_history(valid_time, study_id, lag)`
- `v10_history(valid_time, study_id, lag)`

The `lag` coordinate stores the historical offset in hours. A lag of `0` represents the wind observed at forecast issue time `t`; all other lags represent strictly earlier observations.

The future targets are stored as:

- `u10_target(valid_time, study_id, horizon)`
- `v10_target(valid_time, study_id, horizon)`

where the `horizon` coordinate contains 1, 3, 6, 12, and 24 h.

Using a dedicated lag dimension instead of dozens of independent columns keeps the multi-scale structure explicit and allows the later model to select the short, diurnal, and synoptic branches directly.

Calendar information is encoded cyclically rather than using raw hour and day-of-year integers:

\[
\sin(2\pi h/24),\quad \cos(2\pi h/24)
\]

and

\[
\sin(2\pi d/D),\quad \cos(2\pi d/D),
\]

where `D` is 365 or 366 days depending on the year.

Latitude, longitude, grid identifiers, and basin information remain coordinates and are preserved in the output.


In [12]:
def make_cyclic_time_features(valid_time):
    """
    Create cyclic hour-of-day and day-of-year predictors.

    Parameters
    ----------
    valid_time : xarray.DataArray
        Forecast issue timestamps.

    Returns
    -------
    dict
        Four one-dimensional xarray.DataArray objects:
        hour_sin, hour_cos, doy_sin, and doy_cos.
    """

    times = pd.DatetimeIndex(
        pd.to_datetime(
            valid_time.values
        )
    )

    # -----------------------------------------------------
    # Hour-of-day cycle
    # -----------------------------------------------------

    hour_fraction = (
        times.hour
        + times.minute / 60.0
        + times.second / 3600.0
    )

    hour_angle = (
        2.0
        * np.pi
        * hour_fraction
        / 24.0
    )

    # -----------------------------------------------------
    # Annual cycle
    # -----------------------------------------------------

    days_in_year = np.where(
        times.is_leap_year,
        366.0,
        365.0,
    )

    day_fraction = (
        (times.dayofyear - 1)
        + hour_fraction / 24.0
    )

    year_angle = (
        2.0
        * np.pi
        * day_fraction
        / days_in_year
    )

    coordinates = {
        "valid_time": valid_time.values
    }

    return {
        "hour_sin": xr.DataArray(
            np.sin(hour_angle).astype(np.float32),
            dims=("valid_time",),
            coords=coordinates,
        ),
        "hour_cos": xr.DataArray(
            np.cos(hour_angle).astype(np.float32),
            dims=("valid_time",),
            coords=coordinates,
        ),
        "doy_sin": xr.DataArray(
            np.sin(year_angle).astype(np.float32),
            dims=("valid_time",),
            coords=coordinates,
        ),
        "doy_cos": xr.DataArray(
            np.cos(year_angle).astype(np.float32),
            dims=("valid_time",),
            coords=coordinates,
        ),
    }


def create_vector_features(ds):
    """
    Construct causal multi-scale wind-vector predictors
    and multi-horizon future vector targets.

    Parameters
    ----------
    ds : xarray.Dataset
        Hourly dataset containing u10 and v10 with dimensions
        valid_time × study_id. The dataset may include temporary
        historical and future context around one calendar year.

    Returns
    -------
    xarray.Dataset
        Model-ready vector history, future vector targets,
        cyclic calendar predictors, and preserved coordinates.
    """

    required_variables = {
        "u10",
        "v10",
    }

    missing_variables = (
        required_variables
        .difference(ds.data_vars)
    )

    if missing_variables:
        raise ValueError(
            f"Missing required variables: "
            f"{sorted(missing_variables)}"
        )

    required_dimensions = {
        "valid_time",
        "study_id",
    }

    missing_dimensions = (
        required_dimensions
        .difference(ds.dims)
    )

    if missing_dimensions:
        raise ValueError(
            f"Missing required dimensions: "
            f"{sorted(missing_dimensions)}"
        )

    # -----------------------------------------------------
    # Coordinates for lag and forecast-horizon dimensions
    # -----------------------------------------------------

    lag_coordinate = xr.DataArray(
        np.asarray(
            ALL_LAGS,
            dtype=np.int16
        ),
        dims=("lag",),
        name="lag",
    )

    horizon_coordinate = xr.DataArray(
        np.asarray(
            FORECAST_HORIZONS,
            dtype=np.int16
        ),
        dims=("horizon",),
        name="horizon",
    )

    # -----------------------------------------------------
    # Historical zonal wind tensor
    # -----------------------------------------------------

    u10_history = xr.concat(
        [
            ds["u10"].shift(
                valid_time=lag
            )
            for lag in ALL_LAGS
        ],
        dim=lag_coordinate,
    )

    u10_history = (
        u10_history
        .transpose(
            "valid_time",
            "study_id",
            "lag",
        )
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # Historical meridional wind tensor
    # -----------------------------------------------------

    v10_history = xr.concat(
        [
            ds["v10"].shift(
                valid_time=lag
            )
            for lag in ALL_LAGS
        ],
        dim=lag_coordinate,
    )

    v10_history = (
        v10_history
        .transpose(
            "valid_time",
            "study_id",
            "lag",
        )
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # Future zonal wind targets
    # Negative shift aligns the value at t+h with issue time t.
    # -----------------------------------------------------

    u10_target = xr.concat(
        [
            ds["u10"].shift(
                valid_time=-horizon
            )
            for horizon in FORECAST_HORIZONS
        ],
        dim=horizon_coordinate,
    )

    u10_target = (
        u10_target
        .transpose(
            "valid_time",
            "study_id",
            "horizon",
        )
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # Future meridional wind targets
    # -----------------------------------------------------

    v10_target = xr.concat(
        [
            ds["v10"].shift(
                valid_time=-horizon
            )
            for horizon in FORECAST_HORIZONS
        ],
        dim=horizon_coordinate,
    )

    v10_target = (
        v10_target
        .transpose(
            "valid_time",
            "study_id",
            "horizon",
        )
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # Cyclic calendar predictors at forecast issue time
    # -----------------------------------------------------

    cyclic_features = (
        make_cyclic_time_features(
            ds["valid_time"]
        )
    )

    # -----------------------------------------------------
    # Assemble output dataset
    # -----------------------------------------------------

    feature_ds = xr.Dataset({
        "u10_history": u10_history,
        "v10_history": v10_history,
        "u10_target": u10_target,
        "v10_target": v10_target,
        **cyclic_features,
    })

    # -----------------------------------------------------
    # Identify which temporal scale each lag belongs to
    # -----------------------------------------------------

    feature_ds = (
        feature_ds
        .assign_coords(
            temporal_scale=(
                "lag",
                [
                    LAG_TO_SCALE[int(lag)]
                    for lag in ALL_LAGS
                ],
            )
        )
    )

    # -----------------------------------------------------
    # Variable metadata
    # -----------------------------------------------------

    feature_ds["u10_history"].attrs = {
        "long_name": (
            "Historical 10-m zonal wind component "
            "available at forecast issue time"
        ),
        "units": "m s-1",
    }

    feature_ds["v10_history"].attrs = {
        "long_name": (
            "Historical 10-m meridional wind component "
            "available at forecast issue time"
        ),
        "units": "m s-1",
    }

    feature_ds["u10_target"].attrs = {
        "long_name": (
            "Future 10-m zonal wind-component target"
        ),
        "units": "m s-1",
    }

    feature_ds["v10_target"].attrs = {
        "long_name": (
            "Future 10-m meridional wind-component target"
        ),
        "units": "m s-1",
    }

    feature_ds["lag"].attrs = {
        "long_name": (
            "Hours before forecast issue time"
        ),
        "units": "h",
    }

    feature_ds["horizon"].attrs = {
        "long_name": (
            "Forecast lead time"
        ),
        "units": "h",
    }

    feature_ds["valid_time"].attrs = {
        "long_name": (
            "Forecast issue time"
        )
    }

    # -----------------------------------------------------
    # Dataset-level metadata
    # -----------------------------------------------------

    feature_ds.attrs = {
        "forecast_problem": (
            "multi-horizon 10-m wind-vector forecasting"
        ),
        "target_components": "u10,v10",
        "forecast_horizons_hours": (
            ",".join(
                map(
                    str,
                    FORECAST_HORIZONS
                )
            )
        ),
        "historical_lags_hours": (
            ",".join(
                map(
                    str,
                    ALL_LAGS
                )
            )
        ),
        "maximum_history_hours": (
            MAX_HISTORY_HOURS
        ),
        "maximum_forecast_hours": (
            MAX_FORECAST_HOURS
        ),
        "feature_engineering_version": (
            "vector_multiscale_v1"
        ),
    }

    return feature_ds


## 5. Dry run and temporal-alignment test

Before processing all 16 years, the feature constructor is tested on a small section of the 2005 dataset.

The test explicitly verifies both directions of time alignment:

- a historical feature at lag `k` must equal the source wind component at `t-k`,
- a target at horizon `h` must equal the source wind component at `t+h`.

This is a critical leakage-prevention check because the new dataset contains both historical predictors and future targets.


In [13]:
# ---------------------------------------------------------
# Dry run on the first 400 hours of 2005
# ---------------------------------------------------------

sample_file = YEARLY_DIR / "2005.nc"

with xr.open_dataset(sample_file) as ds_sample:

    sample_input = (
        ds_sample
        .isel(
            valid_time=slice(
                0,
                400
            )
        )
        .load()
    )


sample_features = create_vector_features(
    sample_input
)


print("Dry-run dataset:")
print(sample_features)

print("\nDimensions:")
print(dict(sample_features.sizes))

print("\nData variables:")
print(list(sample_features.data_vars))

print("\nLag coordinates:")
print(sample_features["lag"].values)

print("\nTemporal-scale coordinates:")
print(sample_features["temporal_scale"].values)

print("\nForecast horizons:")
print(sample_features["horizon"].values)


# ---------------------------------------------------------
# Choose an issue time with full past and future context
# ---------------------------------------------------------

test_time_index = 200
test_study_id = 0

test_time = pd.Timestamp(
    sample_input["valid_time"]
    .values[test_time_index]
)


print("\nTest forecast issue time:")
print(test_time)


# ---------------------------------------------------------
# Check historical alignment
# ---------------------------------------------------------

for lag in ALL_LAGS:

    engineered_u = float(
        sample_features["u10_history"]
        .sel(lag=lag)
        .isel(
            valid_time=test_time_index,
            study_id=test_study_id
        )
        .values
    )

    source_u = float(
        sample_input["u10"]
        .isel(
            valid_time=(
                test_time_index
                - lag
            ),
            study_id=test_study_id
        )
        .values
    )

    engineered_v = float(
        sample_features["v10_history"]
        .sel(lag=lag)
        .isel(
            valid_time=test_time_index,
            study_id=test_study_id
        )
        .values
    )

    source_v = float(
        sample_input["v10"]
        .isel(
            valid_time=(
                test_time_index
                - lag
            ),
            study_id=test_study_id
        )
        .values
    )

    assert np.isclose(
        engineered_u,
        source_u
    )

    assert np.isclose(
        engineered_v,
        source_v
    )


# ---------------------------------------------------------
# Check future-target alignment
# ---------------------------------------------------------

for horizon in FORECAST_HORIZONS:

    engineered_u = float(
        sample_features["u10_target"]
        .sel(horizon=horizon)
        .isel(
            valid_time=test_time_index,
            study_id=test_study_id
        )
        .values
    )

    source_u = float(
        sample_input["u10"]
        .isel(
            valid_time=(
                test_time_index
                + horizon
            ),
            study_id=test_study_id
        )
        .values
    )

    engineered_v = float(
        sample_features["v10_target"]
        .sel(horizon=horizon)
        .isel(
            valid_time=test_time_index,
            study_id=test_study_id
        )
        .values
    )

    source_v = float(
        sample_input["v10"]
        .isel(
            valid_time=(
                test_time_index
                + horizon
            ),
            study_id=test_study_id
        )
        .values
    )

    assert np.isclose(
        engineered_u,
        source_u
    )

    assert np.isclose(
        engineered_v,
        source_v
    )


print(
    "\nHistorical alignment: PASSED"
)

print(
    "Future-target alignment: PASSED"
)

print(
    "Dry run: PASSED"
)


Dry-run dataset:
<xarray.Dataset> Size: 26MB
Dimensions:         (valid_time: 400, study_id: 512, lag: 11, horizon: 5)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 3kB 2005-01-01 ... 2005-01-17...
    latitude        (study_id) float64 4kB 22.75 22.75 23.0 ... 30.0 30.0 30.25
    longitude       (study_id) float64 4kB 59.5 59.75 59.25 ... 49.75 50.0 49.0
  * study_id        (study_id) int64 4kB 0 1 2 3 4 5 ... 506 507 508 509 510 511
    grid_id         (study_id) int64 4kB 594 595 578 579 580 581 ... 5 6 7 8 9 1
  * lag             (lag) int16 22B 0 1 2 3 6 12 24 48 72 120 168
  * horizon         (horizon) int16 10B 1 3 6 12 24
    temporal_scale  (lag) <U8 352B 'short' 'short' ... 'synoptic' 'synoptic'
Data variables:
    u10_history     (valid_time, study_id, lag) float32 9MB 3.677 nan ... 6.59
    v10_history     (valid_time, study_id, lag) float32 9MB -5.22 nan ... -3.831
    u10_target      (valid_time, study_id, horizon) float32 4MB 4.085 ... nan
    v10_target  

## 6. Generate annual vector-feature datasets with cross-year continuity

The multi-scale historical predictors require up to 168 h of data before the issue time, while the multi-horizon targets require up to 24 h after the issue time.

Therefore each calendar year is temporarily combined with:

- the final 168 h of the previous year, when available,
- the first 24 h of the following year, when available.

This prevents artificial breaks at January 1 and December 31.

For 2005, data from 2004 are unavailable, so the first 168 issue times are removed.

For 2020, data from 2021 are unavailable, so the final 24 issue times are removed.

All intermediate years retain complete calendar-year coverage.

The final saved timestamp always represents the **forecast issue time**, not the target-valid time.


In [14]:
# ---------------------------------------------------------
# Helper for loading only required context
# ---------------------------------------------------------

def load_previous_tail(year):
    """
    Load the final MAX_HISTORY_HOURS of the previous year.
    """

    previous_file = (
        YEARLY_DIR
        / f"{year - 1}.nc"
    )

    with xr.open_dataset(previous_file) as ds_previous:

        previous_tail = (
            ds_previous
            .isel(
                valid_time=slice(
                    -MAX_HISTORY_HOURS,
                    None
                )
            )
            .load()
        )

    return previous_tail


def load_next_head(year):
    """
    Load the first MAX_FORECAST_HOURS of the following year.
    """

    next_file = (
        YEARLY_DIR
        / f"{year + 1}.nc"
    )

    with xr.open_dataset(next_file) as ds_next:

        next_head = (
            ds_next
            .isel(
                valid_time=slice(
                    0,
                    MAX_FORECAST_HOURS
                )
            )
            .load()
        )

    return next_head


# ---------------------------------------------------------
# NetCDF compression settings
# ---------------------------------------------------------

NETCDF_ENCODING = {
    "u10_history": {
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "dtype": "float32",
    },
    "v10_history": {
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "dtype": "float32",
    },
    "u10_target": {
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "dtype": "float32",
    },
    "v10_target": {
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "dtype": "float32",
    },
    "hour_sin": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
    },
    "hour_cos": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
    },
    "doy_sin": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
    },
    "doy_cos": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
    },
}


# ---------------------------------------------------------
# Generate annual vector-feature datasets
# ---------------------------------------------------------

feature_results = []


for year in range(2005, 2021):

    print(
        f"\nCreating vector features for {year} ..."
    )

    current_file = (
        YEARLY_DIR
        / f"{year}.nc"
    )

    output_file = (
        VECTOR_FEATURE_DIR
        / f"{year}.nc"
    )

    # -----------------------------------------------------
    # Load current calendar year
    # -----------------------------------------------------

    with xr.open_dataset(current_file) as ds_current:

        current = (
            ds_current
            .load()
        )

    context_parts = []

    # -----------------------------------------------------
    # Historical context
    # -----------------------------------------------------

    if year > 2005:

        previous_tail = (
            load_previous_tail(
                year
            )
        )

        context_parts.append(
            previous_tail
        )

    # -----------------------------------------------------
    # Current year
    # -----------------------------------------------------

    context_parts.append(
        current
    )

    # -----------------------------------------------------
    # Future context
    # -----------------------------------------------------

    if year < 2020:

        next_head = (
            load_next_head(
                year
            )
        )

        context_parts.append(
            next_head
        )

    # -----------------------------------------------------
    # Concatenate temporal context
    # -----------------------------------------------------

    combined = xr.concat(
        context_parts,
        dim="valid_time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        combine_attrs="override",
    )

    combined = (
        combined
        .sortby("valid_time")
    )

    combined_times = pd.DatetimeIndex(
        pd.to_datetime(
            combined["valid_time"].values
        )
    )

    assert not combined_times.has_duplicates

    # -----------------------------------------------------
    # Create multi-scale predictors and future targets
    # -----------------------------------------------------

    features = create_vector_features(
        combined
    )

    # -----------------------------------------------------
    # Determine valid issue-time interval for this year
    # -----------------------------------------------------

    year_start = pd.Timestamp(
        f"{year}-01-01 00:00:00"
    )

    year_end = pd.Timestamp(
        f"{year}-12-31 23:00:00"
    )

    valid_start = year_start
    valid_end = year_end

    if year == 2005:

        valid_start = (
            valid_start
            + pd.Timedelta(
                hours=MAX_HISTORY_HOURS
            )
        )

    if year == 2020:

        valid_end = (
            valid_end
            - pd.Timedelta(
                hours=MAX_FORECAST_HOURS
            )
        )

    features = features.sel(
        valid_time=slice(
            valid_start,
            valid_end,
        )
    )

    # -----------------------------------------------------
    # Confirm that no edge NaNs remain
    # -----------------------------------------------------

    for variable in [
        "u10_history",
        "v10_history",
        "u10_target",
        "v10_target",
    ]:

        has_missing = bool(
            features[variable]
            .isnull()
            .any()
            .values
        )

        assert not has_missing, (
            f"{variable} contains missing values "
            f"after edge trimming in {year}."
        )

    # -----------------------------------------------------
    # Save annual model-ready dataset
    # -----------------------------------------------------

    features.to_netcdf(
        output_file,
        encoding=NETCDF_ENCODING,
    )

    # -----------------------------------------------------
    # Record processing summary
    # -----------------------------------------------------

    number_of_hours = (
        features.sizes["valid_time"]
    )

    number_of_cells = (
        features.sizes["study_id"]
    )

    feature_results.append({
        "year": year,
        "start_issue_time": (
            pd.Timestamp(
                features["valid_time"]
                .values[0]
            )
        ),
        "end_issue_time": (
            pd.Timestamp(
                features["valid_time"]
                .values[-1]
            )
        ),
        "hours": number_of_hours,
        "study_cells": number_of_cells,
        "grid_time_samples": (
            number_of_hours
            * number_of_cells
        ),
        "historical_lags": len(ALL_LAGS),
        "forecast_horizons": len(FORECAST_HORIZONS),
        "output_file": output_file.name,
    })

    print(
        f"Saved: {output_file.name}"
    )

    print(
        "Issue times:",
        number_of_hours
    )

    print(
        "Grid-time samples:",
        number_of_hours
        * number_of_cells
    )

    # -----------------------------------------------------
    # Release memory before next year
    # -----------------------------------------------------

    del current
    del context_parts
    del combined
    del features

    if "previous_tail" in locals():
        del previous_tail

    if "next_head" in locals():
        del next_head

    gc.collect()


# ---------------------------------------------------------
# Processing summary
# ---------------------------------------------------------

feature_summary = pd.DataFrame(
    feature_results
)


print(
    "\nVector feature-engineering summary:"
)

display(
    feature_summary
)


print(
    "\nAll annual vector-feature datasets "
    "created successfully."
)



Creating vector features for 2005 ...
Saved: 2005.nc
Issue times: 8592
Grid-time samples: 4399104

Creating vector features for 2006 ...
Saved: 2006.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2007 ...
Saved: 2007.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2008 ...
Saved: 2008.nc
Issue times: 8784
Grid-time samples: 4497408

Creating vector features for 2009 ...
Saved: 2009.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2010 ...
Saved: 2010.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2011 ...
Saved: 2011.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2012 ...
Saved: 2012.nc
Issue times: 8784
Grid-time samples: 4497408

Creating vector features for 2013 ...
Saved: 2013.nc
Issue times: 8760
Grid-time samples: 4485120

Creating vector features for 2014 ...
Saved: 2014.nc
Issue times: 8760
Grid-time samples: 4485120

Creating 

,year,start_issue_time,end_issue_time,hours,study_cells,grid_time_samples,historical_lags,forecast_horizons,output_file
0,2005,2005-01-08,2005-12-31 23:00:00,8592,512,4399104,11,5,2005.nc
1,2006,2006-01-01,2006-12-31 23:00:00,8760,512,4485120,11,5,2006.nc
2,2007,2007-01-01,2007-12-31 23:00:00,8760,512,4485120,11,5,2007.nc
3,2008,2008-01-01,2008-12-31 23:00:00,8784,512,4497408,11,5,2008.nc
4,2009,2009-01-01,2009-12-31 23:00:00,8760,512,4485120,11,5,2009.nc
5,2010,2010-01-01,2010-12-31 23:00:00,8760,512,4485120,11,5,2010.nc
6,2011,2011-01-01,2011-12-31 23:00:00,8760,512,4485120,11,5,2011.nc
7,2012,2012-01-01,2012-12-31 23:00:00,8784,512,4497408,11,5,2012.nc
8,2013,2013-01-01,2013-12-31 23:00:00,8760,512,4485120,11,5,2013.nc
9,2014,2014-01-01,2014-12-31 23:00:00,8760,512,4485120,11,5,2014.nc



All annual vector-feature datasets created successfully.


## 7. Save publication-oriented feature metadata

The feature configuration is saved explicitly so that later training, evaluation, and manuscript preparation can reproduce the exact experiment.

Four metadata tables are produced:

- annual vector feature-generation summary,
- temporal-scale/lag schema,
- forecast-horizon schema,
- model-input/output schema.

These files also make the later ablation study easier because each temporal lag is already assigned to a specific branch.


In [15]:
# ---------------------------------------------------------
# 1. Annual feature-generation summary
# ---------------------------------------------------------

FEATURE_SUMMARY_PATH = (
    SUMMARY_DIR
    / "vector_feature_engineering_summary.csv"
)

feature_summary.to_csv(
    FEATURE_SUMMARY_PATH,
    index=False
)


# ---------------------------------------------------------
# 2. Temporal-scale schema
# ---------------------------------------------------------

TEMPORAL_SCALE_SCHEMA_PATH = (
    SUMMARY_DIR
    / "temporal_scale_schema.csv"
)

temporal_scale_rows = []

for scale_name, lags in TEMPORAL_SCALES.items():

    for branch_order, lag in enumerate(
        lags,
        start=1
    ):

        temporal_scale_rows.append({
            "temporal_scale": scale_name,
            "branch_order": branch_order,
            "lag_hours": lag,
            "u_variable": "u10_history",
            "v_variable": "v10_history",
        })


temporal_scale_schema = pd.DataFrame(
    temporal_scale_rows
)

temporal_scale_schema.to_csv(
    TEMPORAL_SCALE_SCHEMA_PATH,
    index=False
)


# ---------------------------------------------------------
# 3. Forecast-horizon schema
# ---------------------------------------------------------

FORECAST_HORIZON_SCHEMA_PATH = (
    SUMMARY_DIR
    / "forecast_horizon_schema.csv"
)

forecast_horizon_schema = pd.DataFrame({
    "horizon_order": range(
        1,
        len(FORECAST_HORIZONS) + 1
    ),
    "horizon_hours": FORECAST_HORIZONS,
    "u_target": [
        "u10_target"
    ] * len(FORECAST_HORIZONS),
    "v_target": [
        "v10_target"
    ] * len(FORECAST_HORIZONS),
})

forecast_horizon_schema.to_csv(
    FORECAST_HORIZON_SCHEMA_PATH,
    index=False
)


# ---------------------------------------------------------
# 4. Model-input/output schema
# ---------------------------------------------------------

MODEL_SCHEMA_PATH = (
    SUMMARY_DIR
    / "vector_model_schema.csv"
)

model_schema = pd.DataFrame([
    {
        "role": "predictor",
        "group": "static",
        "variable": "latitude",
        "dimensions": "study_id",
    },
    {
        "role": "predictor",
        "group": "static",
        "variable": "longitude",
        "dimensions": "study_id",
    },
    {
        "role": "predictor",
        "group": "calendar",
        "variable": "hour_sin",
        "dimensions": "valid_time",
    },
    {
        "role": "predictor",
        "group": "calendar",
        "variable": "hour_cos",
        "dimensions": "valid_time",
    },
    {
        "role": "predictor",
        "group": "calendar",
        "variable": "doy_sin",
        "dimensions": "valid_time",
    },
    {
        "role": "predictor",
        "group": "calendar",
        "variable": "doy_cos",
        "dimensions": "valid_time",
    },
    {
        "role": "predictor",
        "group": "wind_history",
        "variable": "u10_history",
        "dimensions": "valid_time,study_id,lag",
    },
    {
        "role": "predictor",
        "group": "wind_history",
        "variable": "v10_history",
        "dimensions": "valid_time,study_id,lag",
    },
    {
        "role": "target",
        "group": "future_vector",
        "variable": "u10_target",
        "dimensions": "valid_time,study_id,horizon",
    },
    {
        "role": "target",
        "group": "future_vector",
        "variable": "v10_target",
        "dimensions": "valid_time,study_id,horizon",
    },
])

model_schema.to_csv(
    MODEL_SCHEMA_PATH,
    index=False
)


print("Saved metadata:")

print(
    "\n1.",
    FEATURE_SUMMARY_PATH.resolve()
)

print(
    "\n2.",
    TEMPORAL_SCALE_SCHEMA_PATH.resolve()
)

print(
    "\n3.",
    FORECAST_HORIZON_SCHEMA_PATH.resolve()
)

print(
    "\n4.",
    MODEL_SCHEMA_PATH.resolve()
)


print(
    "\nTemporal-scale schema:"
)

display(
    temporal_scale_schema
)


print(
    "\nForecast-horizon schema:"
)

display(
    forecast_horizon_schema
)


print(
    "\nModel schema:"
)

display(
    model_schema
)


Saved metadata:

1. /home/mike/Desktop/wind_paper/outputs/tables/vector_feature_engineering_summary.csv

2. /home/mike/Desktop/wind_paper/outputs/tables/temporal_scale_schema.csv

3. /home/mike/Desktop/wind_paper/outputs/tables/forecast_horizon_schema.csv

4. /home/mike/Desktop/wind_paper/outputs/tables/vector_model_schema.csv

Temporal-scale schema:


,temporal_scale,branch_order,lag_hours,u_variable,v_variable
0,short,1,0,u10_history,v10_history
1,short,2,1,u10_history,v10_history
2,short,3,2,u10_history,v10_history
3,short,4,3,u10_history,v10_history
4,short,5,6,u10_history,v10_history
5,diurnal,1,12,u10_history,v10_history
6,diurnal,2,24,u10_history,v10_history
7,diurnal,3,48,u10_history,v10_history
8,synoptic,1,72,u10_history,v10_history
9,synoptic,2,120,u10_history,v10_history



Forecast-horizon schema:


,horizon_order,horizon_hours,u_target,v_target
0,1,1,u10_target,v10_target
1,2,3,u10_target,v10_target
2,3,6,u10_target,v10_target
3,4,12,u10_target,v10_target
4,5,24,u10_target,v10_target



Model schema:


,role,group,variable,dimensions
0,predictor,static,latitude,study_id
1,predictor,static,longitude,study_id
2,predictor,calendar,hour_sin,valid_time
3,predictor,calendar,hour_cos,valid_time
4,predictor,calendar,doy_sin,valid_time
5,predictor,calendar,doy_cos,valid_time
6,predictor,wind_history,u10_history,"valid_time,study_id,lag"
7,predictor,wind_history,v10_history,"valid_time,study_id,lag"
8,target,future_vector,u10_target,"valid_time,study_id,horizon"
9,target,future_vector,v10_target,"valid_time,study_id,horizon"


## 8. Final integrity check of the engineered vector datasets

All 16 saved annual vector-feature files are reopened and validated.

The final check verifies that:

- the complete 2005–2020 output sequence exists,
- every dataset contains 512 study-grid cells,
- lag and horizon coordinates exactly match the declared schema,
- all required predictor and target variables are present,
- issue timestamps are unique, hourly, and chronological,
- no missing values remain in model predictors or targets,
- 2005 correctly excludes its unavailable 168-h historical edge,
- 2020 correctly excludes its unavailable 24-h future edge,
- vector-history and future-target alignment agree with the source `u10` and `v10` fields at an independent mid-year timestamp.

This check is deliberately strict because incorrect time alignment would invalidate the forecasting experiment.


In [16]:
# ---------------------------------------------------------
# Final integrity check
# ---------------------------------------------------------

vector_feature_files = sorted(
    VECTOR_FEATURE_DIR.glob("*.nc")
)

expected_years = list(
    range(2005, 2021)
)


assert len(vector_feature_files) == 16

assert [
    int(file.stem)
    for file in vector_feature_files
] == expected_years


REQUIRED_DATA_VARIABLES = [
    "u10_history",
    "v10_history",
    "u10_target",
    "v10_target",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
]


REQUIRED_COORDINATES = [
    "valid_time",
    "study_id",
    "grid_id",
    "latitude",
    "longitude",
    "lag",
    "horizon",
]


def expected_hours_in_year(year):
    """
    Return the number of hourly timestamps in one calendar year.
    """

    start = pd.Timestamp(
        f"{year}-01-01 00:00:00"
    )

    end = pd.Timestamp(
        f"{year + 1}-01-01 00:00:00"
    )

    return int(
        (end - start)
        / pd.Timedelta(hours=1)
    )


integrity_results = []


for file in vector_feature_files:

    year = int(file.stem)

    print(
        f"Checking {year} ..."
    )

    with xr.open_dataset(file) as ds:

        # -------------------------------------------------
        # Expected annual issue-time count
        # -------------------------------------------------

        expected_hours = (
            expected_hours_in_year(
                year
            )
        )

        if year == 2005:

            expected_hours -= (
                MAX_HISTORY_HOURS
            )

        if year == 2020:

            expected_hours -= (
                MAX_FORECAST_HOURS
            )

        assert (
            ds.sizes["valid_time"]
            == expected_hours
        )

        assert (
            ds.sizes["study_id"]
            == 512
        )

        assert (
            ds.sizes["lag"]
            == len(ALL_LAGS)
        )

        assert (
            ds.sizes["horizon"]
            == len(FORECAST_HORIZONS)
        )

        # -------------------------------------------------
        # Required variables
        # -------------------------------------------------

        for variable in REQUIRED_DATA_VARIABLES:

            assert variable in ds.data_vars, (
                f"{variable} missing in {year}"
            )

        # -------------------------------------------------
        # Required coordinates
        # -------------------------------------------------

        for coordinate in REQUIRED_COORDINATES:

            assert coordinate in ds.coords, (
                f"{coordinate} missing in {year}"
            )

        # -------------------------------------------------
        # Exact lag/horizon schemas
        # -------------------------------------------------

        assert np.array_equal(
            ds["lag"].values,
            np.asarray(
                ALL_LAGS,
                dtype=ds["lag"].dtype
            )
        )

        assert np.array_equal(
            ds["horizon"].values,
            np.asarray(
                FORECAST_HORIZONS,
                dtype=ds["horizon"].dtype
            )
        )

        # -------------------------------------------------
        # Time continuity
        # -------------------------------------------------

        times = pd.DatetimeIndex(
            pd.to_datetime(
                ds["valid_time"].values
            )
        )

        assert not times.has_duplicates
        assert times.is_monotonic_increasing

        time_differences = (
            pd.Series(times)
            .diff()
            .dropna()
        )

        assert (
            time_differences
            == pd.Timedelta(hours=1)
        ).all()

        # -------------------------------------------------
        # Missing values
        # -------------------------------------------------

        total_missing = 0

        for variable in REQUIRED_DATA_VARIABLES:

            missing = int(
                ds[variable]
                .isnull()
                .sum()
                .values
            )

            total_missing += missing

            assert missing == 0, (
                f"{variable} contains "
                f"{missing} missing values "
                f"in {year}"
            )

        # -------------------------------------------------
        # Source-alignment check at a safe mid-year time
        # -------------------------------------------------

        check_time = pd.Timestamp(
            f"{year}-07-15 12:00:00"
        )

        check_study_id = 0

        with xr.open_dataset(
            YEARLY_DIR / f"{year}.nc"
        ) as source_ds:

            for lag in ALL_LAGS:

                source_time = (
                    check_time
                    - pd.Timedelta(
                        hours=lag
                    )
                )

                expected_u = float(
                    source_ds["u10"]
                    .sel(
                        valid_time=source_time
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                expected_v = float(
                    source_ds["v10"]
                    .sel(
                        valid_time=source_time
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                actual_u = float(
                    ds["u10_history"]
                    .sel(
                        valid_time=check_time,
                        lag=lag,
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                actual_v = float(
                    ds["v10_history"]
                    .sel(
                        valid_time=check_time,
                        lag=lag,
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                assert np.isclose(
                    actual_u,
                    expected_u
                )

                assert np.isclose(
                    actual_v,
                    expected_v
                )

            for horizon in FORECAST_HORIZONS:

                source_time = (
                    check_time
                    + pd.Timedelta(
                        hours=horizon
                    )
                )

                expected_u = float(
                    source_ds["u10"]
                    .sel(
                        valid_time=source_time
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                expected_v = float(
                    source_ds["v10"]
                    .sel(
                        valid_time=source_time
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                actual_u = float(
                    ds["u10_target"]
                    .sel(
                        valid_time=check_time,
                        horizon=horizon,
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                actual_v = float(
                    ds["v10_target"]
                    .sel(
                        valid_time=check_time,
                        horizon=horizon,
                    )
                    .isel(
                        study_id=check_study_id
                    )
                    .values
                )

                assert np.isclose(
                    actual_u,
                    expected_u
                )

                assert np.isclose(
                    actual_v,
                    expected_v
                )

        # -------------------------------------------------
        # Record integrity result
        # -------------------------------------------------

        integrity_results.append({
            "year": year,
            "hours": ds.sizes["valid_time"],
            "study_cells": ds.sizes["study_id"],
            "lags": ds.sizes["lag"],
            "horizons": ds.sizes["horizon"],
            "grid_time_samples": (
                ds.sizes["valid_time"]
                * ds.sizes["study_id"]
            ),
            "missing_values": total_missing,
            "status": "PASSED",
        })


integrity_summary = pd.DataFrame(
    integrity_results
)


INTEGRITY_SUMMARY_PATH = (
    SUMMARY_DIR
    / "vector_feature_integrity_summary.csv"
)

integrity_summary.to_csv(
    INTEGRITY_SUMMARY_PATH,
    index=False
)


print(
    "\nFinal vector-feature integrity summary:"
)

display(
    integrity_summary
)


print(
    "\nFinal vector-feature integrity check: PASSED"
)

print(
    "\nSaved:"
)

print(
    INTEGRITY_SUMMARY_PATH.resolve()
)

print(
    "\n03_feature_engineering.ipynb: COMPLETED"
)


Checking 2005 ...
Checking 2006 ...
Checking 2007 ...
Checking 2008 ...
Checking 2009 ...
Checking 2010 ...
Checking 2011 ...
Checking 2012 ...
Checking 2013 ...
Checking 2014 ...
Checking 2015 ...
Checking 2016 ...
Checking 2017 ...
Checking 2018 ...
Checking 2019 ...
Checking 2020 ...

Final vector-feature integrity summary:


,year,hours,study_cells,lags,horizons,grid_time_samples,missing_values,status
0,2005,8592,512,11,5,4399104,0,PASSED
1,2006,8760,512,11,5,4485120,0,PASSED
2,2007,8760,512,11,5,4485120,0,PASSED
3,2008,8784,512,11,5,4497408,0,PASSED
4,2009,8760,512,11,5,4485120,0,PASSED
5,2010,8760,512,11,5,4485120,0,PASSED
6,2011,8760,512,11,5,4485120,0,PASSED
7,2012,8784,512,11,5,4497408,0,PASSED
8,2013,8760,512,11,5,4485120,0,PASSED
9,2014,8760,512,11,5,4485120,0,PASSED



Final vector-feature integrity check: PASSED

Saved:
/home/mike/Desktop/wind_paper/outputs/tables/vector_feature_integrity_summary.csv

03_feature_engineering.ipynb: COMPLETED


## Notebook 03 complete

The project now contains a causal, multi-scale, multi-horizon wind-vector feature set.

The next notebook should **not** train the final neural network yet. The next scientific step is to define and generate the Shamal regime and transition labels from a literature-supported meteorological definition. After that, the chronological train/validation/test split and train-only scaling pipeline can be updated for the new vector tensors.
